## Reproduction with Self-developed Package - DIM 

written by **Jinwoo Lee**            
jil527@ucsd.edu | jinwoo-lee.com

Dec, 2025       
as a PSYC201A's **Reproduction Project**

**Note:** This script aims to reproduce the key findings of Kim & Kim (2022) with the authors' original data and our own package - `DIM`. 

Since **DIM** is part of an ongoing research project in our lab, the corresponding GitHub repository is not publicly available.

---
### Step 1. Loading the Packages
In addition to basic packages such as pandas, I will import my DIM package. `Level1` class is specifically designed for performing conventional inter-subject representational similarity analysis (IS-RSA; e.g., Anna Karenina Modeling).    

When running a traditional IS-RSA with a `Level1` model, the following parameters should be defined first:

- **model**: Choose one of the two theoretically defined IS-RSA models; ['AnnaK' or 'NearestN']  
  - `AnnaK`: assuming that participants with higher trait anxiety scores will show (dis)similar brain morphology.
  - `NearestN`: assuming that participants with (dis)similar triat anxiety scores will show (dis)similar brain morphology.      
  *Example:* Kim & Kim (2022) tested the model using Anna Karenina → `AnnaK`

- **distance**: Define which metric to calculate the relationship between participants in the test domain (in our case, trait anxiety). Available metrics depend on the model.  
  - if `model == 'AnnaK'`: ['Mean' or 'Min']  
  - if `model == 'NearestN'`: ['Euclidean' or 'Cosine']       
  *Example:* Kim & Kim (2022) used the mean metric with the AnnaK model → `Mean`

- **weighting**: When there are two or more input variables in the test domain, choose how to weight each variable when calculating participant relationships; ['None' or 'Learn']  
  - `None`: weighting all X features equally
  - `Learn`: optimizing the weight of each X feature to maximize the correlation between the test and target domains    
  *Example:* Kim & Kim (2022) used only one variable (STAI total score), so no weighting was applied → `None`

- **dependency**: Choose the metric to calculate the correlation between domains; ['Pearson' or 'Spearman']  
  - `Pearson`: Pearson’s r  
  - `Spearman`: Spearman’s rho       
  *Example:* Kim & Kim (2022) calculated the correlation between STAI and brain distance matrices using Spearman’s rho → `Spearman`

The defined Level1 model gets **(n_subjects * n_features)** shaped test domain dataframe and **(n_subjects * n_subjects)** shaped distance matrix of the target domain. In my case, I will input (119, 1) shaped STAI total score vector and (119, 119) shaped brain morphology distance matrix. After that, `DIM`'s Level1 model will compute the distance matrix of STAI scores based on the specified IS-RSA model and paramters, and then calculate the correlation between the two distance matrices.

Please see 'Step 4' for the usage of `Level1` class. 

In [1]:
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, '../DIM/src')
from lib.dim.level1 import Level1  # importing our DIM package
from scipy.spatial.distance import pdist, squareform

---
### Step 2. Preparing a STAI Score Vector as a DIM's Input

I will extract the STAI scores of each group as a 1D array, which will be used as one of the inputs for DIM. To do this:

1. Using `Meta.csv`, I will filter the participants based on the same three criteria used by the authors: age, availability of MRI data, and past/present psychiatric diagnosis. 

2. I will then extract the STAI summary scores of the participants. 

> **[Caution]** The participant IDs in `Meta.csv` are listed randomly, without ascending or descending order. The brain morphology data is sorted in ascending order of IDs. To align the participant order in STAI with brain dissimilarity amtrix, you must use `sort_values` to sort IDs in ascending order. Failing to do so may misalign the STAI distance matrix with the brain distance matrix, preventing meaningful relationships from being captured.

In [2]:
meta = pd.read_csv('../data-from-authors/Meta.csv')
STAI = pd.read_csv('../data-from-authors/STAI_G_X2.csv')

### ==================================================================
### PART I. YOUNG ADULTS GROUP 
# age
youth = meta[meta['Age'].isin(['20-25', '25-30', '30-35'])]

# IDP availability
exclude_subjects = ['sub-032339', 'sub-032341', 'sub-032459', 'sub-032370', 
                    'sub-032466', 'sub-032438', 'sub-032509']
youth = youth[~youth['Unnamed: 0'].isin(exclude_subjects)]

# SKID Diagnoses
youth['SKID_Diagnoses'] = pd.Categorical(youth['SKID_Diagnoses'])
youth['SKID_Diagnoses_numeric'] = youth['SKID_Diagnoses'].cat.codes + 1 
Hyouth = youth[(youth['SKID_Diagnoses_numeric'] == 0) | (youth['SKID_Diagnoses_numeric'] == 10)]

# leaving only relevant info
columns_to_drop = list(range(3, 14)) + list(range(15, 21))  
Hyouth = Hyouth.drop(Hyouth.columns[columns_to_drop], axis=1)

# merging STAI data with Hyouth
Hyouth = pd.merge(Hyouth, STAI, on = 'Unnamed: 0')

# IMPORTANT: Matching the order of subjects with brain morphology data
Hyouth = Hyouth.sort_values('Unnamed: 0').reset_index(drop = True)

# constructing the young samples' STAI scores array
Hyouth_anx = Hyouth[['STAI_Trait_Anxiety']].to_numpy()


### ==================================================================
### PART II. OLDER ADULTS GROUP 
# age
senior = meta[meta['Age'].isin(['60-65', '65-70', '70-75'])]

# IDP availability
senior_exclude_subjects = ["sub-032339", "sub-032341", "sub-032459", "sub-032370",
                           "sub-032466", "sub-032438", "sub-032509", "sub-032392", "sub-032443", "sub-032488"]
senior = senior[~senior['Unnamed: 0'].isin(senior_exclude_subjects)]

# SKID Diagnosis
senior['SKID_Diagnoses'] = pd.Categorical(senior['SKID_Diagnoses'])
senior['SKID_Diagnoses_numeric'] = senior ['SKID_Diagnoses'].cat.codes + 1 

Hsenior = senior[(senior['SKID_Diagnoses_numeric'] == 5) | (youth['SKID_Diagnoses_numeric'] == 6)]

# leaving only relevant info
Hsenior = Hsenior.drop(Hsenior.columns[columns_to_drop], axis=1)

# merging STAI data with Hsenior
Hsenior = pd.merge(Hsenior, STAI, on = 'Unnamed: 0')

# IMPORTANT: Matching the order of subjects with brain morphology data
Hsenior = Hsenior.sort_values('Unnamed: 0').reset_index(drop = True)

# Constructing the Young Samples' STAI Scores Array
Hsenior_anx = Hsenior[['STAI_Trait_Anxiety']].to_numpy()

/var/folders/pk/pd0z53jd7rb6v8r41_lcv3tm0000gp/T/ipykernel_61687/3370226118.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  Hsenior = senior[(senior['SKID_Diagnoses_numeric'] == 5) | (youth['SKID_Diagnoses_numeric'] == 6)]


---
### Step 3. Constructing a Brain Morphology Distance Matrix as DIM's Input

Unlike the STAI scores, the Brain Morphology data for the target domain in DIM must be provided as a distance matrix. To do this, we will use the shared data preprocessed with the 5% slice-level thresholding provided by the authors.      

Since this data is in `(n_features, n_subjects)` format, we first need to transpose it before calculating the distance matrix. We will use `pdist` and `squareform` from `scipy.spatial.distance` for this purpose.


In [3]:
### ==================================================================
### PART I. YOUNG ADULTS GROUP 
yh_prob_5p_L = pd.read_csv('../data-from-authors/tractfiles/yh_probmap_5p_L.csv')
yh_prob_5p_L = yh_prob_5p_L.iloc[:, 1:]  # Remove the first column (index)

Hyouth_brain = squareform(pdist(yh_prob_5p_L.T, metric = 'euclidean'))

### ==================================================================
### PART II. OLDER ADULTS GROUP 
oh_prob_5p_L = pd.read_csv('../data-from-authors/tractfiles/oh_prob_5p_L_new.csv')
oh_prob_5p_L = oh_prob_5p_L.iloc[:, 1:]  # Remove the first column (index)

Hsenior_brain = squareform(pdist(oh_prob_5p_L.T, metric = 'euclidean'))

---
### Step 4. Anna Karenina Model Testing with DIM's Level1 Class
Using `_anx` and `_brain` prepared in Steps 2 and 3 as the main inputs for the `Level1` class, I will conduct Anna Karenina Model testing and examine whether the statistics can be reproduced. This process proceeds in three main stages:

1. **Model specification:** Set the parameters required by the `Level1` model, such as `model`, `distance`, `weighting`, and `dependency` (see *Overview of DIM Package*).      

2. **Model fitting:** Use the `fit` method of the `Level1` class to fit the model with `stai_values` and `HAnnaKBrain` as inputs. During this process, `Level1` internally generates an Anna-K distance matrix of the same 119×119 size as `HAnnaKBrain` from `stai_values`.       

3. **Permutation test:** Use the `permutation_test` method of the `Level1` class to assess the statistical significance of the correlation between the two distance matrices. The authors’ R-based `vegan::mantel` function uses one-sided testing (`greater`) for P-value calculation (see [here](https://vegandevs.github.io/vegan/reference/mantel.html)). Accordingly, we also use one-sided testing rather than a two-sided test.  


In [4]:
### ==================================================================
### PART I. YOUNG ADULTS GROUP 
### Defining the Model 
young_ak_model = Level1(
    model = "AnnaK",
    distance = "Mean",
    weighting = "None",
    dependency = "spearman_r",
    normalize = False # not normalizing as in the original study
)

### Fitting the Model
young_ak_model.fit(Hyouth_anx, Hyouth_brain)

### Conducting Permutation Test
young_ak_mantel = young_ak_model.permutation_test_discovery(
    n_perms = 10000,
    verbose = True,
    return_null_dist = True,
    seed = 42
)

young_ak_mantel

Validation successful!


Permutation test (refitting): 100%|██████████| 10000/10000 [00:19<00:00, 508.74perm/s]


{'observed_stat': 0.1482987105846405,
 'p_value': np.float64(0.0426),
 'n_perms': 10000,
 'null_mean': np.float64(-0.0014572440253728474),
 'null_std': np.float64(0.08488472827755539),
 'computation_time': 19.69390606880188,
 'null_distribution': array([-0.05733099, -0.02783272,  0.15806741, ...,  0.04909881,
        -0.06928889,  0.04331321], shape=(10000,))}

In [5]:
### ==================================================================
### PART II. OLDER ADULTS GROUP 
### Defining the Model 
older_ak_model = Level1(
    model = "AnnaK",
    distance = "Mean",
    weighting = "None",
    dependency = "spearman_r",
    normalize = False # not normalizing as in the original study
)

### Fitting the Model
older_ak_model.fit(Hsenior_anx, Hsenior_brain)

### Conducting Permutation Test
older_ak_mantel = older_ak_model.permutation_test_discovery(
    n_perms = 10000,
    verbose = True,
    return_null_dist = True,
    seed = 42
)

older_ak_mantel

Validation successful!


Permutation test (refitting): 100%|██████████| 10000/10000 [00:03<00:00, 3276.13perm/s]


{'observed_stat': 0.2940811216831207,
 'p_value': np.float64(0.0165),
 'n_perms': 10000,
 'null_mean': np.float64(-0.0008054625425669656),
 'null_std': np.float64(0.13677994425369377),
 'computation_time': 3.0554988384246826,
 'null_distribution': array([ 0.02247226,  0.1751264 ,  0.18698882, ..., -0.02548333,
        -0.14252409, -0.02663732], shape=(10000,))}

For visualizing the scatterplot between mean anxiety score and brain morphological dissimilarity value, we saved `D_x_vec` and `D_y_vec` for each group, which are one-dimensional vectors including upper elements of mean anxiety matrix and brain dissimilarity matrix. 

In [6]:
young_D_x = young_ak_model.get_Dx()
young_idx = np.triu_indices(young_D_x.shape[0], k = 1)
young_D_x_vec = young_D_x[young_idx]
young_D_y_vec = Hyouth_brain[young_idx]

df_young = pd.DataFrame({'anx_mean': young_D_x_vec, 'brain_dissimilarity': young_D_y_vec})

older_D_x = older_ak_model.get_Dx()
older_idx = np.triu_indices(older_D_x.shape[0], k = 1)
older_D_x_vec = older_D_x[older_idx]
older_D_y_vec = Hsenior_brain[older_idx]

df_older = pd.DataFrame({'anx_mean': older_D_x_vec, 'brain_dissimilarity': older_D_y_vec})

df_young.to_csv('results/confirmatory/youth_vectors_ours.csv', index = False)
df_older.to_csv('results/confirmatory/older_vectors_ours.csv', index = False)